# Abdomen CT — nnUNet v2 — **Faz 2b: Eğitim + Değerlendirme**

**Zorunlu dataset (Faz2a output'u):**
- `nnunet-preprocessed` → `nnunet_preprocessed/`, `splits_out/`, `nifti_val/` içerir

**Opsiyonel dataset'ler:**
- `nnunet-checkpoint` → önceki session checkpoint (devam için)
- `abdomen` → ham DICOM (yalnızca `nifti_val/` eksikse)

---

| Adım | Süre (T. yakl.) |
|------|------------------|
| nnUNet eğitim 3d_fullres | 8–12 saat |
| Inference + Değerlendirme | 20–30 dk |

---
## 0. Ortam ve GPU

In [ ]:
import os, sys, subprocess
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')
IS_LOCAL  = not IS_KAGGLE and not IS_COLAB
env_name  = 'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'
print(f'Ortam : {env_name}')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı! Kaggle: Settings→Accelerator→GPU')
print(f'GPU   : {torch.cuda.get_device_name(0)}')
print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'CUDA  : {torch.version.cuda}')

---
## 1. Kütüphane Kurulumu

In [ ]:
import importlib, shutil, sysconfig

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'nnunetv2', 'SimpleITK', 'pydicom', 'scipy', 'tqdm'],
    check=True
)
importlib.invalidate_caches()

def find_nnunet_cmd(name: str) -> str:
    p = shutil.which(name)
    if p: return p
    for d in [sysconfig.get_path('scripts'), str(Path(sys.executable).parent),
              '/usr/local/bin', '/opt/conda/bin']:
        c = Path(d) / name
        if c.exists(): return str(c)
    return name

for cmd in ['nnUNetv2_train', 'nnUNetv2_predict']:
    print(f'  {cmd}: {Path(find_nnunet_cmd(cmd)).exists()}')

---
## 2. Konfigürasyon

In [ ]:
import os, json, shutil, time
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List

# ─── Eğitim ayarları ───────────────────────────────────────────────────────
CONTINUE_TRAINING  = False  # True → mevcut checkpoint'ten devam
FOLD               = 0
DATASET_ID         = 100
DATASET_NAME       = 'Abdomen'
GITHUB_URL         = 'https://github.com/ramazan2020/abdomen1.git'

CHECKPOINT_URL = ''   # CONTINUE_TRAINING=True ise önceki session zip URL'si

SUPER_CLASSES: List[str] = [
    'acute_cholecystitis', 'kidney_ureter_stone', 'acute_pancreatitis',
    'aortic_aneurysm_dissection', 'acute_appendicitis', 'acute_diverticulitis',
]

# ─── Kaggle dataset slug'ları ───────────────────────────────────────────────
KAGGLE_DATASET_SLUG = 'abdomen'          # ham DICOM + Bilgi.xlsx
PRE_DATASET_SLUG    = 'abdomen-nnunet'   # Faz2a output — Kaggle'daki slug ile eşleşmeli
# ────────────────────────────────────────────────────────────────────────────

if IS_KAGGLE:
    WORK_DIR  = Path('/kaggle/working')
    TMP_DIR   = Path('/kaggle/tmp')
    _base     = Path('/kaggle/input/datasets/ramazan2020')
    DATA_DIR  = _base / KAGGLE_DATASET_SLUG
    _pre_base = _base / PRE_DATASET_SLUG

    if not _pre_base.exists():
        raise FileNotFoundError(
            f'Zorunlu dataset bulunamadı: {_pre_base}\n'
            f'Sağ panelden "{PRE_DATASET_SLUG}" (Faz2a output) ekleyin.\n'
            f'Dataset slug farklıysa PRE_DATASET_SLUG değişkenini güncelleyin.'
        )

    # Input'taki preprocessed yolu (read-only)
    _pre_input = next(
        (p for p in [_pre_base / 'nnunet_preprocessed', _pre_base]
         if (p / f'Dataset{DATASET_ID}_{DATASET_NAME}').exists()),
        _pre_base / 'nnunet_preprocessed'
    )
    # nnUNet eğitimde .npz yanına .npy yazar → /kaggle/input/ salt okunur olduğu
    # için preprocessed'i /kaggle/tmp/'ya kopyalarız (60 GB, yazılabilir).
    NNUNET_PREPROCESSED_P = TMP_DIR / 'nnunet_preprocessed'

    NIFTI_VAL_DIR = next(
        (p for p in [_pre_base / 'nifti_val']
         if p.exists() and any(p.glob('*.nii.gz'))),
        _pre_base / 'nifti_val'
    )
    EGITIM_DIR  = DATA_DIR / 'Egitim Verisi'
    YARISMA_DIR = DATA_DIR / 'Test Verisi'
    BILGI_XLSX  = DATA_DIR / 'Bilgi.xlsx'
    NIFTI_DIR   = TMP_DIR / 'nifti_val_tmp'
    SPLIT_DIR   = WORK_DIR / 'splits'
    RESULTS_P   = WORK_DIR / 'nnunet_results'

elif IS_COLAB:
    DRIVE_BASE            = Path('/content/drive/MyDrive/Abdomen')
    EGITIM_DIR            = Path('/content/kaggle_data/Egitim Verisi')
    YARISMA_DIR           = Path('/content/kaggle_data/Test Verisi')
    BILGI_XLSX            = Path('/content/kaggle_data/Bilgi.xlsx')
    NIFTI_DIR             = Path('/content/nifti_val_tmp')
    NIFTI_VAL_DIR         = DRIVE_BASE / 'nifti_val'
    SPLIT_DIR             = DRIVE_BASE / 'splits'
    _pre_input            = Path('/content/nnunet_preprocessed')
    NNUNET_PREPROCESSED_P = _pre_input
    RESULTS_P             = Path('/content/nnunet_results')
    WORK_DIR              = Path('/content')

elif IS_LOCAL:
    try:
        from dotenv import load_dotenv; load_dotenv()
    except ImportError: pass
    PROJECT_ROOT          = Path('.').resolve()
    WORK_DIR              = PROJECT_ROOT / 'outputs'
    EGITIM_DIR            = Path(os.environ.get('ABDOMEN_TRAIN_DIR', str(PROJECT_ROOT / 'Egitim Verisi')))
    YARISMA_DIR           = Path(os.environ.get('ABDOMEN_TEST_DIR',  str(PROJECT_ROOT / 'Test Verisi')))
    BILGI_XLSX            = Path(os.environ.get('ABDOMEN_BILGI_XLSX', str(PROJECT_ROOT / 'Bilgi.xlsx')))
    NIFTI_DIR             = WORK_DIR / 'nifti_val_tmp'
    NIFTI_VAL_DIR         = WORK_DIR / 'nifti_val'
    SPLIT_DIR             = WORK_DIR / 'splits'
    _pre_input            = WORK_DIR / 'nnunet_preprocessed'
    NNUNET_PREPROCESSED_P = _pre_input
    RESULTS_P             = WORK_DIR / 'nnunet_results'

RESULTS_P.mkdir(parents=True, exist_ok=True)
NIFTI_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw']          = str(WORK_DIR / 'nnunet_raw_dummy')
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED_P)
os.environ['nnUNet_results']      = str(RESULTS_P)
os.environ['OMP_NUM_THREADS']     = '1'
os.environ['ABDOMEN_SPLIT_DIR']   = str(SPLIT_DIR)
os.environ['ABDOMEN_TRAIN_DIR']   = str(EGITIM_DIR)
os.environ['ABDOMEN_TEST_DIR']    = str(YARISMA_DIR)
os.environ['ABDOMEN_BILGI_XLSX']  = str(BILGI_XLSX)

print(f'Ortam               : {env_name}')
print(f'CONTINUE_TRAINING   : {CONTINUE_TRAINING}')
print(f'PRE_DATASET_SLUG    : {PRE_DATASET_SLUG}')
if IS_KAGGLE:
    print(f'Preprocessed (input): {_pre_input}  (✓={(_pre_input / f"Dataset{DATASET_ID}_{DATASET_NAME}").exists()})')
print(f'nnUNet_preprocessed : {NNUNET_PREPROCESSED_P}  ← yazılabilir hedef')
print(f'SPLIT_DIR           : {SPLIT_DIR}')
print(f'NIFTI_VAL_DIR       : {NIFTI_VAL_DIR}  (✓={NIFTI_VAL_DIR.exists()})')
print(f'RESULTS_P           : {RESULTS_P}')
if IS_KAGGLE:
    print(f'BILGI_XLSX          : {BILGI_XLSX}  (✓={BILGI_XLSX.exists()})')

_plans_input = _pre_input / f'Dataset{DATASET_ID}_{DATASET_NAME}' / 'nnUNetPlans.json'
if not _plans_input.exists():
    raise FileNotFoundError(
        f'nnUNetPlans.json bulunamadı: {_plans_input}\n'
        f'Faz2a\'yı tamamlayıp "{PRE_DATASET_SLUG}" dataset olarak ekleyin.'
    )
print('nnUNetPlans.json    : ✓')

---
## 3. Preprocessed → Tmp (Symlink)

`/kaggle/input/` salt okunur — nnUNet `.npz` yanına `.npy` yazmaya çalışır ve **OSError** verir.  
Çözüm: `/kaggle/tmp/` altında aynı dizin yapısını oluştur, dosyaları symlink'le. Yalnızca `.npy` tmp'ye yazılır.

In [ ]:
if IS_KAGGLE:
    _src_ds = _pre_input / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    _dst_ds = NNUNET_PREPROCESSED_P / f'Dataset{DATASET_ID}_{DATASET_NAME}'

    _npy_existing = list(_dst_ds.rglob('*.npy')) if _dst_ds.exists() else []
    _npz_total    = list(_src_ds.rglob('*.npz'))

    if _dst_ds.exists() and len(_npy_existing) >= len(_npz_total):
        print(f'Symlink+npy zaten hazır: {len(_npy_existing)} .npy, {len(_npz_total)} .npz')
    else:
        print(f'Symlink yapısı oluşturuluyor: {_src_ds} → {_dst_ds}')
        _linked = _skipped = 0
        for src_f in sorted(_src_ds.rglob('*')):
            if not src_f.is_file():
                continue
            rel   = src_f.relative_to(_src_ds)
            dst_f = _dst_ds / rel
            dst_f.parent.mkdir(parents=True, exist_ok=True)
            if dst_f.exists() or dst_f.is_symlink():
                _skipped += 1
                continue
            os.symlink(src_f, dst_f)
            _linked += 1
        print(f'  ✓  {_linked} symlink oluşturuldu, {_skipped} mevcut')

    os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED_P)
    print(f'nnUNet_preprocessed → {NNUNET_PREPROCESSED_P}')
    _free = shutil.disk_usage(str(TMP_DIR)).free / 1e9
    print(f'Tmp serbest         : {_free:.1f} GB  (.npy dosyaları buraya yazılacak)')
else:
    print('Kaggle dışı — symlink atlandı')

---
## 3. GitHub Repo + Splits

In [ ]:
if IS_LOCAL:
    REPO_DIR = Path('.').resolve()
else:
    REPO_DIR = WORK_DIR / 'abdomen1'
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git', 'clone', '--depth=1', GITHUB_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], capture_output=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.splits     import make_splits, raw_case_id
from src.lifting    import BboxLifter
from src.evaluation import f1_at_iou, top5_f1_mean

# ── manifest.csv ─────────────────────────────────────────────────────────────
_manifest_csv = SPLIT_DIR / 'manifest.csv'
if not _manifest_csv.exists():
    print('manifest.csv oluşturuluyor...')
    from src.preprocessing import build_manifest
    build_manifest(_manifest_csv)
    print(f'  ✓  ({_manifest_csv.stat().st_size/1e3:.0f} KB)')
else:
    print(f'manifest.csv mevcut  ({_manifest_csv.stat().st_size/1e3:.0f} KB)')

# ── splits.csv + fold CSV'leri ───────────────────────────────────────────────
_fold_train_csv = SPLIT_DIR / f'fold{FOLD}_train.csv'
_fold_val_csv   = SPLIT_DIR / f'fold{FOLD}_val.csv'
if not _fold_train_csv.exists() or not _fold_val_csv.exists():
    print('Splits oluşturuluyor...')
    make_splits(out_dir=SPLIT_DIR)
    print('  ✓')
else:
    print('splits.csv mevcut')

# load_fold() src/config.py'deki modül-seviye SPLIT_DIR'ı kullandığı için
# env var değişikliğini görmez; doğrudan CSV okuması güvenlidir.
manifest    = pd.read_csv(_manifest_csv)
train_cases = pd.read_csv(_fold_train_csv)['Case Number'].astype(str).tolist()
val_cases   = pd.read_csv(_fold_val_csv)['Case Number'].astype(str).tolist()
all_cases   = sorted(set(train_cases + val_cases))

print(f'Manifest  : {len(manifest):,} satır')
print(f'Fold {FOLD} : {len(train_cases)} train + {len(val_cases)} val = {len(all_cases)} toplam')

def lift_2d_to_3d(manifest_df, case, delta_z=2, iou_th=0.3):
    return BboxLifter(manifest_df, egitim_dir=EGITIM_DIR,
                      yarisma_dir=YARISMA_DIR, delta_z=delta_z, iou_th=iou_th).lift(case)

---
## 4. Checkpoint İndirme / Geri Yükleme

`CONTINUE_TRAINING = True` ise:
1. `CHECKPOINT_URL` doluysa → zip indirilir, `/kaggle/working/nnunet_results/` altına extract edilir
2. URL boşsa → `/kaggle/working/nnunet_results/` içinde mevcut checkpoint aranır

**URL nasıl alınır?**  
Önceki session'da **Save Version** sonrası `Output` sekmesinden `nnunet_results/` klasörünü zip olarak indirin ve direkt link kopyalayın.

In [ ]:
import zipfile, urllib.request

_ckpt_dst = RESULTS_P / f'Dataset{DATASET_ID}_{DATASET_NAME}'

if not CONTINUE_TRAINING:
    print('Yeni eğitim (CONTINUE_TRAINING=False)')

elif CHECKPOINT_URL:
    # URL'den indirip extract et
    _zip_path = TMP_DIR / 'nnunet_checkpoint.zip' if IS_KAGGLE else WORK_DIR / 'nnunet_checkpoint.zip'
    print(f'Checkpoint indiriliyor: {CHECKPOINT_URL}')
    urllib.request.urlretrieve(CHECKPOINT_URL, str(_zip_path))
    print(f'  İndirildi ({_zip_path.stat().st_size / 1e6:.0f} MB), extract ediliyor...')
    with zipfile.ZipFile(str(_zip_path), 'r') as zf:
        # Zip içinde nnunet_results/ veya Dataset100_Abdomen/ yapısı olabilir
        _names = zf.namelist()
        _has_results_root = any(n.startswith('nnunet_results/') for n in _names)
        if _has_results_root:
            # nnunet_results/Dataset100_Abdomen/... → WORK_DIR/nnunet_results/...
            zf.extractall(str(WORK_DIR))
        else:
            # Dataset100_Abdomen/... → RESULTS_P/Dataset100_Abdomen/...
            zf.extractall(str(RESULTS_P))
    _zip_path.unlink(missing_ok=True)
    print(f'  ✓  Checkpoint: {_ckpt_dst}  (exists={_ckpt_dst.exists()})')

else:
    # URL yok — mevcut RESULTS_P içinde checkpoint var mı kontrol et
    _fold_dir = _ckpt_dst / 'nnUNetTrainer__nnUNetPlans__3d_fullres' / f'fold_{FOLD}'
    _has_ckpt = any((_fold_dir).glob('checkpoint_*.pth')) if _fold_dir.exists() else False
    if _has_ckpt:
        print(f'Mevcut checkpoint bulundu: {_fold_dir}  → devam edilecek')
    else:
        print(f'⚠ CONTINUE_TRAINING=True fakat checkpoint bulunamadı: {_fold_dir}')
        print('  CHECKPOINT_URL doldurun veya CONTINUE_TRAINING=False yapın')

---
## 5. Eğitim

**Kaggle 9 saat session sınırı:** Checkpoint otomatik kaydedilir.  
Session bitince: çıktıyı `nnunet-checkpoint` dataset olarak yayınlayın, yeni session'da `CONTINUE_TRAINING = True` yapın.

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU gerekli!'

_args = [find_nnunet_cmd('nnUNetv2_train'),
         str(DATASET_ID), '3d_fullres', str(FOLD), '--npz']
if CONTINUE_TRAINING:
    _args.append('--c')

print(f'=== nnUNet Eğitim ===')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'Dataset : {DATASET_ID}  Config: 3d_fullres  Fold: {FOLD}  Devam: {CONTINUE_TRAINING}')
print(f'Results : {RESULTS_P}')
print('─' * 50)

_proc = subprocess.Popen(
    _args, env={**os.environ},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    universal_newlines=True
)
for line in _proc.stdout:
    print(line, end='', flush=True)
_proc.wait()

print('─' * 50)
if _proc.returncode == 0:
    print('Eğitim tamamlandı ✓')
else:
    print(f'Eğitim kodu: {_proc.returncode} (session kesildi / hata)')
    print('Checkpoint nnunet_results/ altında — Save Version ile kaydedin')

if IS_COLAB:
    _src = RESULTS_P / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    _dst = Path('/content/drive/MyDrive/Abdomen/nnunet_results') / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    if _src.exists():
        _dst.parent.mkdir(parents=True, exist_ok=True)
        if _dst.exists(): shutil.rmtree(str(_dst))
        shutil.copytree(str(_src), str(_dst))
        print(f"Drive'a yedeklendi: {_dst}")

---
## 6. Val NIfTI Hazırlama

In [ ]:
import SimpleITK as sitk
import pydicom
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm

VAL_INPUT_DIR = WORK_DIR / 'val_input'
VAL_INPUT_DIR.mkdir(parents=True, exist_ok=True)

_linked = 0
_missing_cases = []

for c in val_cases:
    raw = raw_case_id(c)
    dst = VAL_INPUT_DIR / f'case_{raw:05d}_0000.nii.gz'
    if dst.exists(): continue
    src = NIFTI_VAL_DIR / f'case_{raw:05d}_0000.nii.gz'
    if src.exists():
        try: os.symlink(src, dst)
        except (OSError, NotImplementedError): shutil.copy2(str(src), str(dst))
        _linked += 1
    else:
        _missing_cases.append(c)

print(f'Faz2a nifti_val: {_linked} linklendi  |  eksik: {len(_missing_cases)}')

if _missing_cases and EGITIM_DIR.exists():
    print(f'{len(_missing_cases)} vaka DICOM\'dan üretiliyor...')

    def _dicom_to_nifti(case: str) -> str:
        raw = raw_case_id(case)
        out = NIFTI_DIR / f'case_{raw:05d}_0000.nii.gz'
        if out.exists(): return 'skip'
        case_dir = EGITIM_DIR / str(raw)
        if not case_dir.exists(): return f'missing:{case}'
        reader = sitk.ImageSeriesReader()
        names  = reader.GetGDCMSeriesFileNames(str(case_dir))
        if not names: return f'no_dcm:{case}'
        size_map = {}
        for n in names:
            try:
                h = pydicom.dcmread(n, stop_before_pixels=True)
                size_map.setdefault((int(h.Rows), int(h.Columns)), []).append(n)
            except Exception: pass
        if len(size_map) > 1:
            names = max(size_map.values(), key=len)
        reader.SetFileNames(names)
        try:
            sitk.WriteImage(reader.Execute(), str(out))
            return 'ok'
        except Exception as e:
            return f'err:{e}'

    sitk.ProcessObject.GlobalWarningDisplayOff()
    with ThreadPoolExecutor(max_workers=4) as ex:
        _res = list(tqdm(ex.map(_dicom_to_nifti, _missing_cases),
                         total=len(_missing_cases), desc='Val DICOM→NIfTI'))
    sitk.ProcessObject.GlobalWarningDisplayOn()

    for c in _missing_cases:
        raw = raw_case_id(c)
        src = NIFTI_DIR / f'case_{raw:05d}_0000.nii.gz'
        dst = VAL_INPUT_DIR / f'case_{raw:05d}_0000.nii.gz'
        if src.exists() and not dst.exists():
            try: os.symlink(src, dst)
            except (OSError, NotImplementedError): shutil.copy2(str(src), str(dst))
elif _missing_cases:
    print(f'⚠ {len(_missing_cases)} NIfTI eksik ve ham DICOM dataset yok')
    print(f'  Sağ panelden "{KAGGLE_DATASET_SLUG}" ekleyin')

_ready = len(list(VAL_INPUT_DIR.glob('*.nii.gz')))
print(f'Val input hazır: {_ready}/{len(val_cases)} dosya')

---
## 7. Inference — Validasyon Seti

In [ ]:
VAL_OUTPUT_DIR = (RESULTS_P
                  / f'Dataset{DATASET_ID}_{DATASET_NAME}'
                  / 'nnUNetTrainer__nnUNetPlans__3d_fullres'
                  / f'fold_{FOLD}'
                  / 'val_predictions')
VAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Val input : {len(list(VAL_INPUT_DIR.glob("*.nii.gz")))} dosya')
print('Inference başlatılıyor...')

_proc = subprocess.Popen(
    [find_nnunet_cmd('nnUNetv2_predict'),
     '-i', str(VAL_INPUT_DIR), '-o', str(VAL_OUTPUT_DIR),
     '-d', str(DATASET_ID), '-c', '3d_fullres', '-f', str(FOLD),
     '--save_probabilities'],
    env={**os.environ},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    universal_newlines=True
)
for line in _proc.stdout:
    print(line, end='', flush=True)
_proc.wait()
print(f'Inference tamamlandı: {len(list(VAL_OUTPUT_DIR.glob("*.nii.gz")))} mask')

---
## 8. Değerlendirme — F1 Skoru

In [ ]:
from scipy import ndimage

def seg_to_pred_df(pred_dir: Path) -> pd.DataFrame:
    rows = []
    for nii_path in sorted(pred_dir.glob('*.nii.gz')):
        try: raw = int(nii_path.stem.split('_')[1])
        except: continue
        mask      = sitk.GetArrayFromImage(sitk.ReadImage(str(nii_path)))
        prob_path = nii_path.with_suffix('').with_suffix('.npz')
        probs     = np.load(str(prob_path))['probabilities'] if prob_path.exists() else None
        for label_id in range(1, len(SUPER_CLASSES)+1):
            binary = (mask == label_id).astype(np.uint8)
            if binary.sum() == 0: continue
            labeled, n_comp = ndimage.label(binary)
            total_vox = float(binary.sum())
            for comp_id in range(1, n_comp+1):
                cm = (labeled == comp_id)
                coords = np.where(cm)
                z1,z2 = int(coords[0].min()), int(coords[0].max())
                y1,y2 = int(coords[1].min()), int(coords[1].max())
                x1,x2 = int(coords[2].min()), int(coords[2].max())
                score = float(probs[label_id][cm].mean()) if probs is not None else float(cm.sum())/total_vox
                for z in range(z1, z2+1):
                    rows.append({'case':raw,'image_id':z,'class':label_id-1,'score':score,
                                 'x1':float(x1),'y1':float(y1),'x2':float(x2),'y2':float(y2)})
    return pd.DataFrame(rows)

pred_df = seg_to_pred_df(VAL_OUTPUT_DIR)
print(f'Prediction: {len(pred_df):,} satır, {pred_df["case"].nunique() if not pred_df.empty else 0} vaka')

gt_rows = []
for case in val_cases:
    raw = raw_case_id(case)
    for b in lift_2d_to_3d(manifest, case):
        for z in range(int(b['z1']), int(b['z2'])+1):
            gt_rows.append({'case':raw,'image_id':z,'class':int(b['class']),
                            'x1':float(b['x1']),'y1':float(b['y1']),
                            'x2':float(b['x2']),'y2':float(b['y2'])})
gt_df = pd.DataFrame(gt_rows)

if pred_df.empty:
    print('Prediction boş — inference adımını kontrol edin')
else:
    top5   = top5_f1_mean(pred_df, gt_df)
    detail = f1_at_iou(pred_df, gt_df, iou_th=0.3)

    print(f'\n=== nnUNet v2 — Fold {FOLD} ===')
    print(f'Top-5 Mean F1 : {top5["top5_mean_f1"]:.4f}')
    print(f'Macro F1 @0.3 : {detail["macro_f1"]:.4f}')
    print(f'Micro F1 @0.3 : {detail["micro_f1"]:.4f}')
    print()
    for cls_id, cls_name in enumerate(SUPER_CLASSES):
        if cls_id in detail['per_class']:
            m = detail['per_class'][cls_id]
            print(f'  {cls_id} {cls_name:<30} P={m["precision"]:.3f} R={m["recall"]:.3f} F1={m["f1"]:.3f}')

---
## 9. Sonuçları Kaydet

In [ ]:
OUTPUT_DIR = WORK_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not pred_df.empty:
    pred_df.to_csv(OUTPUT_DIR / f'pred_fold{FOLD}.csv', index=False)
    gt_df.to_csv(OUTPUT_DIR   / f'gt_fold{FOLD}.csv',   index=False)
    summary = {
        'model': 'nnUNet_3d_fullres', 'fold': FOLD, 'val_cases': len(val_cases),
        'top5_mean_f1': top5['top5_mean_f1'],
        'macro_f1_03' : detail['macro_f1'],
        'micro_f1_03' : detail['micro_f1'],
        'per_threshold': {str(th): float(f1v) for th, f1v in top5['per_threshold']},
    }
    (OUTPUT_DIR / f'summary_fold{FOLD}.json').write_text(json.dumps(summary, indent=2, default=float))

print(f'Çıktılar: {OUTPUT_DIR}')
for f in sorted(OUTPUT_DIR.glob('*')):
    print(f'  {f.name}  ({f.stat().st_size/1e3:.0f} KB)')

if IS_COLAB:
    _dst = Path('/content/drive/MyDrive/Abdomen/output_nnunet')
    if _dst.exists(): shutil.rmtree(str(_dst))
    shutil.copytree(str(OUTPUT_DIR), str(_dst))
    print(f"Drive'a kopyalandı: {_dst}")

print('\nTamamlandı!')
if IS_KAGGLE:
    print('Save Version → checkpoint + sonuçlar /kaggle/working/ altında saklanır')